# Obstacle Detection Pipeline Test

The pipeline logic lives in the **`obstacle_detection`** package, so this
notebook and the Gradio app share ONE implementation (no copy-paste drift).

| Module | Responsibility |
|--------|----------------|
| `config.py` | server paths + all tunable thresholds (`ObstacleConfig`) |
| `models.py` | load + cache SAM / vehicle / DA3 (`get_models`, `reset_models`) |
| `car.py` | pick the car SAM mask (best overlap with the vehicle bbox) |
| `detector.py` | obstacle-decision logic (`find_obstacles`) + `run_pipeline` |
| `visualize.py` | 5-panel result figure (`render_result_figure`) |

**Logic**
1. Run SAM2 + vehicle detection + DA3 on the image.
2. Pick the **car mask**: the SAM2 mask that overlaps the **vehicle bbox** the most.
3. `car_depth` = average depth of the car mask -> the foreground/background reference. Background masks (farther than the car) are ignored.
4. For each remaining mask compute its average depth and a vertical depth **delta** (top half vs bottom half) — the delta flags the receding **ground** plane so it is not mistaken for an obstacle.
5. The car's own sub-parts (windows / wheels / doors) sit at the **same depth** as the car, so the foreground test drops them automatically.
6. A **foreground** mask (closer than the car) that overlaps the car mask by **>= 5%** is an **obstacle**. Several masks can qualify.

Depth note: `da3.depth` is metric (smaller = closer), so *foreground = smaller depth than the car*. The depth-map panel shows the inverse (close = brighter).

The last panel paints the **car mask cyan** and every **obstacle mask red**.

To tune thresholds, edit `obstacle_detection/config.py`, or pass an override per call:
`run_pipeline(path, config=ObstacleConfig(min_car_overlap=0.10))`.

In [ ]:
import os
import matplotlib.pyplot as plt

from obstacle_detection import (
    run_pipeline,
    render_result_figure,
    ObstacleConfig,
    OBSTACLE_IMAGES_DIR,
    reset_models,   # call reset_models() to force a fresh model load without a kernel restart
)

In [ ]:
os.listdir(OBSTACLE_IMAGES_DIR)

In [ ]:
# Run the pipeline on a test image (change the filename to try others).
test_image_path = os.path.join(OBSTACLE_IMAGES_DIR, "9b8f8464-4921-4ba7-8778-b96fcd174517.png")

# To experiment with thresholds without editing config.py, pass an override, e.g.:
#   result = run_pipeline(test_image_path, config=ObstacleConfig(min_car_adjacency=0.08))
result = run_pipeline(test_image_path)
print(f"Final Pipeline Result: {result['obstacle_exist']}")

# 5-panel view: Original | SAM | Vehicle | Depth | Obstacle Mask
# (same renderer the Gradio app uses).
fig_img = render_result_figure(result)
plt.figure(figsize=(25, 8))
plt.imshow(fig_img)
plt.axis("off")
plt.show()